In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

In [20]:
RESULTS_PATH = Path("../data/raw/IF_1872_2026/results.csv")

results = pd.read_csv(RESULTS_PATH)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [21]:
print(results.info())
print("---" * 30)
print(results.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB
None
------------------------------------------------------------------------------------------
date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64


In [22]:
results["date"] = pd.to_datetime(results["date"], format="%Y-%m-%d", errors="raise")

results.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [23]:
TEAM_NAME_MAPPING = {
    "Turkey": "Türkiye",
    "United States": "USA",
    "Czech Republic": "Czechia",
}

results[["home_team", "away_team"]] = results[["home_team", "away_team"]].replace(TEAM_NAME_MAPPING)

In [24]:
teams_after_cleaning = set(results["home_team"]).union(results["away_team"])

[name for name in ["Turkey", "United States", "Czech Republic"] if name in teams_after_cleaning]

[]

In [25]:
[name for name in ["Türkiye", "USA", "Czechia"] if name in teams_after_cleaning]

['Türkiye', 'USA', 'Czechia']

In [26]:
score_columns = ["home_score", "away_score"]

complete_scores = results[score_columns].notna().all(axis=1)
missing_scores = results[score_columns].isna().all(axis=1)
partial_scores = results[score_columns].isna().any(axis=1) & ~missing_scores

assert not partial_scores.any(), "Rows with exactly one missing score found"

historical_matches = results.loc[complete_scores].copy()
future_matches = results.loc[missing_scores].copy()

print("historical_matches:", len(historical_matches))
print("future_matches:", len(future_matches))

historical_matches: 49433
future_matches: 44


In [27]:
historical_matches[score_columns] = historical_matches[score_columns].astype(int)

historical_matches[score_columns].dtypes

home_score    int64
away_score    int64
dtype: object

In [28]:
historical_matches["outcome"] = np.select(
    [
        historical_matches["home_score"] > historical_matches["away_score"],
        historical_matches["home_score"] < historical_matches["away_score"],
    ],
    ["home_win", "away_win"],
    default="draw",
)

historical_matches[["home_team", "away_team", "home_score", "away_score", "outcome"]].head()

,home_team,away_team,home_score,away_score,outcome
0,Scotland,England,0,0,draw
1,England,Scotland,4,2,home_win
2,Scotland,England,2,1,home_win
3,England,Scotland,2,2,draw
4,Scotland,England,3,0,home_win


In [29]:
historical_matches["outcome"].value_counts()

outcome
home_win    24227
away_win    13963
draw        11243
Name: count, dtype: int64

In [30]:
all_output_teams = (
    set(historical_matches["home_team"])
    | set(historical_matches["away_team"])
    | set(future_matches["home_team"])
    | set(future_matches["away_team"])
)

old_names = {"Turkey", "United States", "Czech Republic"}
canonical_names = {"Türkiye", "USA", "Czechia"}
valid_outcomes = {"home_win", "away_win", "draw"}

assert len(historical_matches) == 49_433
assert len(future_matches) == 44
assert len(historical_matches) + len(future_matches) == len(results)

assert historical_matches[score_columns].notna().all().all()
assert future_matches[score_columns].isna().all().all()
assert historical_matches[score_columns].dtypes.eq("int64").all()

assert historical_matches["outcome"].isin(valid_outcomes).all()
assert old_names.isdisjoint(all_output_teams), f"Old names still present: {old_names & all_output_teams}"
assert canonical_names.issubset(all_output_teams), f"Missing canonical names: {canonical_names - all_output_teams}"

print("All checks passed.")

All checks passed.


## Elo ratings cleaning exploration

In [42]:
ELO_PATH = Path("../data/raw/FIFA_WK_Elo_Ratings/elo_ratings_wc2026.csv")

elo = pd.read_csv(ELO_PATH)

elo.head(10)

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0
5,1902,1902-12-31,Austria,4,AT,1914,4,1914,4,1914,...,1,0,0,1,0,0,5,0,UEFA,0
6,1902,1902-12-31,Uruguay,6,UY,1879,4,1879,5,1879,...,1,0,0,0,1,0,0,6,CONMEBOL,0
7,1903,1903-12-31,Argentina,1,AR,2005,1,2021,1,2018,...,1,1,0,1,1,0,8,3,CONMEBOL,0
8,1903,1903-12-31,England,2,EN,1968,1,2079,2,1989,...,42,38,0,49,17,14,273,108,UEFA,0
9,1903,1903-12-31,Scotland,3,SQ,1956,1,2104,2,2015,...,40,42,0,58,10,14,296,109,UEFA,0


In [41]:
print(elo.info())
print("---" * 30)
print(elo.isna().sum())
print("---" * 30)
print(elo.columns)
print("---" * 30)
print(elo["country"].nunique())


<class 'pandas.DataFrame'>
RangeIndex: 4683 entries, 0 to 4682
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   year             4683 non-null   int64
 1   snapshot_date    4683 non-null   str  
 2   country          4683 non-null   str  
 3   rank             4683 non-null   int64
 4   country_code     4683 non-null   str  
 5   rating           4683 non-null   int64
 6   rank_max         4683 non-null   int64
 7   rating_max       4683 non-null   int64
 8   rank_avg         4683 non-null   int64
 9   rating_avg       4683 non-null   int64
 10  rank_min         4683 non-null   int64
 11  rating_min       4683 non-null   int64
 12  matches_total    4683 non-null   int64
 13  matches_home     4683 non-null   int64
 14  matches_away     4683 non-null   int64
 15  matches_neutral  4683 non-null   int64
 16  wins             4683 non-null   int64
 17  losses           4683 non-null   int64
 18  draws            46

In [43]:
sorted(elo["country"].unique())


['Algeria',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bosnia and Herzegovina',
 'Brazil',
 'Canada',
 'Cape Verde',
 'Colombia',
 'Croatia',
 'Curaçao',
 'Czechia',
 'DR Congo',
 'Ecuador',
 'Egypt',
 'England',
 'France',
 'Germany',
 'Ghana',
 'Haiti',
 'Iran',
 'Iraq',
 'Ivory Coast',
 'Japan',
 'Jordan',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Norway',
 'Panama',
 'Paraguay',
 'Portugal',
 'Qatar',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'South Africa',
 'South Korea',
 'Spain',
 'Sweden',
 'Switzerland',
 'Tunisia',
 'Turkey',
 'United States',
 'Uruguay',
 'Uzbekistan']

In [45]:
elo.dtypes

year               int64
snapshot_date        str
country              str
rank               int64
country_code         str
rating             int64
rank_max           int64
rating_max         int64
rank_avg           int64
rating_avg         int64
rank_min           int64
rating_min         int64
matches_total      int64
matches_home       int64
matches_away       int64
matches_neutral    int64
wins               int64
losses             int64
draws              int64
goals_for          int64
goals_against      int64
confederation        str
is_host            int64
dtype: object

In [46]:
elo.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,4683.0,NaN,NaN,NaN,1975.612428,32.255221,1901.0,1951.0,1978.0,2003.0,2026.0
snapshot_date,4683,127,1992-12-31,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,4683,48,England,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rank,4683.0,NaN,NaN,NaN,38.580397,33.021081,1.0,13.0,30.0,55.0,188.0
country_code,4683,48,EN,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,4683.0,NaN,NaN,NaN,1692.405296,210.380644,973.0,1568.0,1693.0,1848.0,2213.0
rank_max,4683.0,NaN,NaN,NaN,18.203288,22.281645,1.0,3.0,11.0,25.0,153.0
rating_max,4683.0,NaN,NaN,NaN,1828.045697,223.437088,1085.0,1680.0,1835.0,2013.0,2223.0
rank_avg,4683.0,NaN,NaN,NaN,36.3754,30.728601,1.0,13.0,28.0,47.0,159.0
rating_avg,4683.0,NaN,NaN,NaN,1660.898142,205.305537,1016.0,1566.0,1660.0,1797.5,2023.0


In [47]:
elo["country"].nunique(), sorted(elo["country"].unique())

(48,
 ['Algeria',
  'Argentina',
  'Australia',
  'Austria',
  'Belgium',
  'Bosnia and Herzegovina',
  'Brazil',
  'Canada',
  'Cape Verde',
  'Colombia',
  'Croatia',
  'Curaçao',
  'Czechia',
  'DR Congo',
  'Ecuador',
  'Egypt',
  'England',
  'France',
  'Germany',
  'Ghana',
  'Haiti',
  'Iran',
  'Iraq',
  'Ivory Coast',
  'Japan',
  'Jordan',
  'Mexico',
  'Morocco',
  'Netherlands',
  'New Zealand',
  'Norway',
  'Panama',
  'Paraguay',
  'Portugal',
  'Qatar',
  'Saudi Arabia',
  'Scotland',
  'Senegal',
  'South Africa',
  'South Korea',
  'Spain',
  'Sweden',
  'Switzerland',
  'Tunisia',
  'Turkey',
  'United States',
  'Uruguay',
  'Uzbekistan'])

In [48]:
elo["year"].min(), elo["year"].max(), sorted(elo["year"].unique())

(np.int64(1901),
 np.int64(2026),
 [np.int64(1901),
  np.int64(1902),
  np.int64(1903),
  np.int64(1904),
  np.int64(1905),
  np.int64(1906),
  np.int64(1907),
  np.int64(1908),
  np.int64(1909),
  np.int64(1910),
  np.int64(1911),
  np.int64(1912),
  np.int64(1913),
  np.int64(1914),
  np.int64(1915),
  np.int64(1916),
  np.int64(1917),
  np.int64(1918),
  np.int64(1919),
  np.int64(1920),
  np.int64(1921),
  np.int64(1922),
  np.int64(1923),
  np.int64(1924),
  np.int64(1925),
  np.int64(1926),
  np.int64(1927),
  np.int64(1928),
  np.int64(1929),
  np.int64(1930),
  np.int64(1931),
  np.int64(1932),
  np.int64(1933),
  np.int64(1934),
  np.int64(1935),
  np.int64(1936),
  np.int64(1937),
  np.int64(1938),
  np.int64(1939),
  np.int64(1940),
  np.int64(1941),
  np.int64(1942),
  np.int64(1943),
  np.int64(1944),
  np.int64(1945),
  np.int64(1946),
  np.int64(1947),
  np.int64(1948),
  np.int64(1949),
  np.int64(1950),
  np.int64(1951),
  np.int64(1952),
  np.int64(1953),
  np.int64(1

In [49]:
elo["snapshot_date"].nunique(), sorted(elo["snapshot_date"].unique())[:20], sorted(elo["snapshot_date"].unique())[-20:]

(127,
 ['1901-12-31',
  '1902-12-31',
  '1903-12-31',
  '1904-12-31',
  '1905-12-31',
  '1906-12-31',
  '1907-12-31',
  '1908-12-31',
  '1909-12-31',
  '1910-12-31',
  '1911-12-31',
  '1912-12-31',
  '1913-12-31',
  '1914-12-31',
  '1915-12-31',
  '1916-12-31',
  '1917-12-31',
  '1918-12-31',
  '1919-12-31',
  '1920-12-31'],
 ['2008-12-31',
  '2009-12-31',
  '2010-12-31',
  '2011-12-31',
  '2012-12-31',
  '2013-12-31',
  '2014-12-31',
  '2015-12-31',
  '2016-12-31',
  '2017-12-31',
  '2018-12-31',
  '2019-12-31',
  '2020-12-31',
  '2021-12-31',
  '2022-12-31',
  '2023-12-31',
  '2024-12-31',
  '2025-12-31',
  '2026-05-27',
  '2026-12-31'])

In [50]:
pd.to_datetime(elo["snapshot_date"], errors="coerce").isna().sum()

np.int64(0)

In [51]:
elo.assign(snapshot_date_parsed=pd.to_datetime(elo["snapshot_date"], errors="coerce"))[
    ["year", "snapshot_date", "snapshot_date_parsed"]
].drop_duplicates().sort_values(["year", "snapshot_date"]).tail(30)

,year,snapshot_date,snapshot_date_parsed
3243,1998,1998-12-31,1998-12-31
3291,1999,1999-12-31,1999-12-31
3339,2000,2000-12-31,2000-12-31
3387,2001,2001-12-31,2001-12-31
3435,2002,2002-12-31,2002-12-31
3483,2003,2003-12-31,2003-12-31
3531,2004,2004-12-31,2004-12-31
3579,2005,2005-12-31,2005-12-31
3627,2006,2006-12-31,2006-12-31
3675,2007,2007-12-31,2007-12-31


In [52]:
elo.duplicated().sum()

np.int64(0)

In [53]:
elo.duplicated(subset=["country", "year", "snapshot_date"]).sum()

np.int64(0)

In [54]:
elo[elo.duplicated(subset=["country", "year", "snapshot_date"], keep=False)].sort_values(
    ["country", "year", "snapshot_date"]
)

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host


In [55]:
elo.groupby(["country", "year"]).size().value_counts().sort_index()

1    4587
2      48
Name: count, dtype: int64

In [56]:
elo.groupby(["country", "year"]).size().reset_index(name="snapshot_count").query(
    "snapshot_count > 1"
).sort_values(["country", "year"]).head(100)

,country,year,snapshot_count
94,Algeria,2026,2
219,Argentina,2026,2
324,Australia,2026,2
449,Austria,2026,2
572,Belgium,2026,2
607,Bosnia and Herzegovina,2026,2
720,Brazil,2026,2
823,Canada,2026,2
891,Cape Verde,2026,2
980,Colombia,2026,2


In [57]:
elo.groupby("country")["year"].agg(["min", "max", "count", "nunique"]).sort_values(
    ["count", "country"]
)

,min,max,count,nunique
country,,,,
Bosnia and Herzegovina,1992,2026,36,35
Uzbekistan,1992,2026,36,35
Croatia,1940,2026,43,42
Qatar,1970,2026,58,57
Cape Verde,1959,2026,69,68
Senegal,1959,2026,69,68
Ghana,1957,2026,71,70
Morocco,1957,2026,71,70
Saudi Arabia,1957,2026,71,70


In [58]:
elo.groupby("country")["snapshot_date"].max().sort_values()

country
Algeria                   2026-12-31
Mexico                    2026-12-31
Morocco                   2026-12-31
Netherlands               2026-12-31
New Zealand               2026-12-31
Norway                    2026-12-31
Panama                    2026-12-31
Paraguay                  2026-12-31
Portugal                  2026-12-31
Qatar                     2026-12-31
Jordan                    2026-12-31
Saudi Arabia              2026-12-31
Senegal                   2026-12-31
South Africa              2026-12-31
South Korea               2026-12-31
Spain                     2026-12-31
Sweden                    2026-12-31
Switzerland               2026-12-31
Tunisia                   2026-12-31
Turkey                    2026-12-31
United States             2026-12-31
Scotland                  2026-12-31
Japan                     2026-12-31
Ivory Coast               2026-12-31
Iraq                      2026-12-31
Argentina                 2026-12-31
Australia                 2026

In [59]:
latest_per_country = elo.loc[
    pd.to_datetime(elo["snapshot_date"]).groupby(elo["country"]).idxmax()
].sort_values("country")

latest_per_country[["country", "year", "snapshot_date", "rank", "rating", "confederation", "is_host"]]

,country,year,snapshot_date,rank,rating,confederation,is_host
4614,Algeria,2026,2026-12-31,35,1743,CAF,0
4588,Argentina,2026,2026-12-31,2,2113,CONMEBOL,0
4610,Australia,2026,2026-12-31,26,1783,AFC,0
4607,Austria,2026,2026-12-31,23,1827,UEFA,0
4604,Belgium,2026,2026-12-31,19,1867,UEFA,0
4626,Bosnia and Herzegovina,2026,2026-12-31,66,1594,UEFA,0
4591,Brazil,2026,2026-12-31,5,1984,CONMEBOL,0
4609,Canada,2026,2026-12-31,25,1784,CONCACAF,1
4629,Cape Verde,2026,2026-12-31,72,1549,CAF,0
4593,Colombia,2026,2026-12-31,7,1975,CONMEBOL,0


In [60]:
latest_per_country["snapshot_date"].value_counts().sort_index()

snapshot_date
2026-12-31    48
Name: count, dtype: int64

In [61]:
latest_per_country.shape

(48, 23)

In [62]:
results_hist = pd.read_csv("../data/processed/results_historical.csv")
results_future = pd.read_csv("../data/processed/results_future.csv")
wc_teams = pd.read_csv("../data/raw/FIFA_WC_1930_2026/wc_2026_teams.csv")
wc_fixtures = pd.read_csv("../data/raw/FIFA_WC_1930_2026/wc_2026_fixtures.csv")

elo_countries = set(elo["country"])
results_countries = set(results_hist["home_team"]) | set(results_hist["away_team"]) | set(results_future["home_team"]) | set(results_future["away_team"])
wc_team_countries = set(wc_teams["team"])

print("Elo not in wc_2026_teams:")
print(sorted(elo_countries - wc_team_countries))

print("wc_2026_teams not in Elo:")
print(sorted(wc_team_countries - elo_countries))

print("Elo not in results:")
print(sorted(elo_countries - results_countries))

print("World Cup teams not in results:")
print(sorted(wc_team_countries - results_countries))

Elo not in wc_2026_teams:
['Turkey', 'United States']
wc_2026_teams not in Elo:
['Türkiye', 'USA']
Elo not in results:
['Turkey', 'United States']
World Cup teams not in results:
[]


In [63]:
wc_fixtures.head()

,group,stage,team1,team2,venue,city,country,date,kickoff_et,team1_confederation,team1_fifa_rank,team1_coach,team2_confederation,team2_fifa_rank,team2_coach
0,A,Group Stage,Mexico,South Africa,Estadio Azteca,Mexico City,Mexico,2026-06-11,20:00 ET,CONCACAF,15.0,Javier Aguirre,CAF,60.0,Hugo Broos
1,A,Group Stage,South Korea,Czechia,Estadio Akron,Guadalajara,Mexico,2026-06-11,22:00 ET,AFC,25.0,Hong Myung-bo,UEFA,41.0,Ivan Hasek
2,A,Group Stage,South Korea,Mexico,Estadio Akron,Guadalajara,Mexico,2026-06-18,21:00 ET,AFC,25.0,Hong Myung-bo,CONCACAF,15.0,Javier Aguirre
3,A,Group Stage,Czechia,South Africa,Estadio Akron,Guadalajara,Mexico,2026-06-18,18:00 ET,UEFA,41.0,Ivan Hasek,CAF,60.0,Hugo Broos
4,A,Group Stage,Czechia,Mexico,Estadio Azteca,Mexico City,Mexico,2026-06-24,21:00 ET,UEFA,41.0,Ivan Hasek,CONCACAF,15.0,Javier Aguirre


In [64]:
wc_fixtures.columns

Index(['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date',
       'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach',
       'team2_confederation', 'team2_fifa_rank', 'team2_coach'],
      dtype='str')

In [65]:
for col in wc_fixtures.columns:
    if "team" in col.lower() or "home" in col.lower() or "away" in col.lower():
        print(col, wc_fixtures[col].dropna().nunique())
        print(sorted(wc_fixtures[col].dropna().unique())[:100])
        print()

team1 80
['1A', '1B', '1C', '1D', '1E', '1F', '1G', '1H', '1I', '1J', '1K', '1L', '3rd Place', 'Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Best 3rd #1', 'Best 3rd #3', 'Best 3rd #5', 'Best 3rd #7', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czechia', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'Finalist 1', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'QF1', 'QF3', 'QF5', 'QF7', 'Qatar', 'R32 W1', 'R32 W11', 'R32 W13', 'R32 W15', 'R32 W3', 'R32 W5', 'R32 W7', 'R32 W9', 'SF1', 'SF3', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Türkiye', 'USA', 'Uruguay', 'Uzbekistan']

team2 80
['2A', '2B', '2C', '2D', '2E', '2F', '2G', '2H', '2I', '2J', '2K', '2L', '3rd Place', 'Algeria', 'Argentina', 'Australia', 'Austria', 'Bel

In [66]:
ELO_NAME_MAPPING = {
    "Turkey": "Türkiye",
    "United States": "USA",
}

elo_clean = elo.copy()

elo_clean["snapshot_date"] = pd.to_datetime(
    elo_clean["snapshot_date"],
    format="%Y-%m-%d",
    errors="raise",
)

elo_clean["is_host"] = elo_clean["is_host"].astype(bool)

elo_clean["country"] = elo_clean["country"].replace(ELO_NAME_MAPPING)

elo_clean = elo_clean.sort_values(
    ["country", "snapshot_date"]
).reset_index(drop=True)

In [72]:
AS_OF_DATE = pd.Timestamp("2026-05-27")

elo_latest = (
    elo_clean.loc[elo_clean["snapshot_date"] <= AS_OF_DATE]
    .sort_values(["country", "snapshot_date"])
    .groupby("country", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

elo_latest.sort_values("rating", ascending=False)[
    ["country", "snapshot_date", "rating", "rank", "confederation", "is_host"]
].head(15)

,country,snapshot_date,rating,rank,confederation,is_host
40,Spain,2026-05-27,2165,1,UEFA,False
1,Argentina,2026-05-27,2113,2,CONMEBOL,False
17,France,2026-05-27,2081,3,UEFA,False
16,England,2026-05-27,2020,4,UEFA,False
33,Portugal,2026-05-27,1984,5,UEFA,False
6,Brazil,2026-05-27,1984,5,CONMEBOL,False
9,Colombia,2026-05-27,1975,7,CONMEBOL,False
28,Netherlands,2026-05-27,1961,8,UEFA,False
14,Ecuador,2026-05-27,1933,9,CONMEBOL,False
10,Croatia,2026-05-27,1930,10,UEFA,False


In [73]:
assert len(elo_clean) == 4_683
assert elo_clean["country"].nunique() == 48
assert elo_clean.isna().sum().sum() == 0
assert elo_clean.duplicated().sum() == 0
assert elo_clean.duplicated(subset=["country", "year", "snapshot_date"]).sum() == 0
assert (elo_clean["year"] == elo_clean["snapshot_date"].dt.year).all()

assert {"Turkey", "United States"}.isdisjoint(set(elo_clean["country"]))
assert {"Türkiye", "USA"}.issubset(set(elo_clean["country"]))

assert len(elo_latest) == 48
assert elo_latest["country"].nunique() == 48
assert elo_latest.duplicated("country").sum() == 0
assert elo_latest["snapshot_date"].max() <= AS_OF_DATE

print("Elo cleaning checks passed.")

Elo cleaning checks passed.


## Fixtures dataset


In [83]:
FIXTURES_PATH = Path("../data/raw/FIFA_WC_1930_2026/wc_2026_fixtures.csv")
WC_TEAMS_PATH = Path("../data/raw/FIFA_WC_1930_2026/wc_2026_teams.csv")
ELO_LATEST_PATH = Path("../data/processed/elo_latest.csv")

fixtures = pd.read_csv(FIXTURES_PATH)
wc_teams = pd.read_csv(WC_TEAMS_PATH)
elo_latest = pd.read_csv(ELO_LATEST_PATH)

fixtures.head()

,group,stage,team1,team2,venue,city,country,date,kickoff_et,team1_confederation,team1_fifa_rank,team1_coach,team2_confederation,team2_fifa_rank,team2_coach
0,A,Group Stage,Mexico,South Africa,Estadio Azteca,Mexico City,Mexico,2026-06-11,20:00 ET,CONCACAF,15.0,Javier Aguirre,CAF,60.0,Hugo Broos
1,A,Group Stage,South Korea,Czechia,Estadio Akron,Guadalajara,Mexico,2026-06-11,22:00 ET,AFC,25.0,Hong Myung-bo,UEFA,41.0,Ivan Hasek
2,A,Group Stage,South Korea,Mexico,Estadio Akron,Guadalajara,Mexico,2026-06-18,21:00 ET,AFC,25.0,Hong Myung-bo,CONCACAF,15.0,Javier Aguirre
3,A,Group Stage,Czechia,South Africa,Estadio Akron,Guadalajara,Mexico,2026-06-18,18:00 ET,UEFA,41.0,Ivan Hasek,CAF,60.0,Hugo Broos
4,A,Group Stage,Czechia,Mexico,Estadio Azteca,Mexico City,Mexico,2026-06-24,21:00 ET,UEFA,41.0,Ivan Hasek,CONCACAF,15.0,Javier Aguirre


In [84]:
print(fixtures.shape)
print("---" * 25)
print(fixtures.info())
print("---" * 25)
print(fixtures.columns)
print("---" * 25)
print(fixtures.tail(10))

(104, 15)
---------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   group                72 non-null     str    
 1   stage                104 non-null    str    
 2   team1                104 non-null    str    
 3   team2                104 non-null    str    
 4   venue                104 non-null    str    
 5   city                 104 non-null    str    
 6   country              104 non-null    str    
 7   date                 104 non-null    str    
 8   kickoff_et           104 non-null    str    
 9   team1_confederation  72 non-null     str    
 10  team1_fifa_rank      72 non-null     float64
 11  team1_coach          72 non-null     str    
 12  team2_confederation  72 non-null     str    
 13  team2_fifa_rank      72 non-null     float64
 14  team2_coach    

In [85]:
fixtures["date"] = pd.to_datetime(
    fixtures["date"],
    format="%Y-%m-%d",
    errors="raise",
)

print("min date:", fixtures["date"].min())
print("max date:", fixtures["date"].max())
print("date dtype:", fixtures["date"].dtype)

min date: 2026-06-11 00:00:00
max date: 2026-07-19 00:00:00
date dtype: datetime64[us]


In [86]:
qualified_teams = set(wc_teams["team"])
elo_teams = set(elo_latest["country"])

fixtures["team1_is_known"] = fixtures["team1"].isin(qualified_teams)
fixtures["team2_is_known"] = fixtures["team2"].isin(qualified_teams)
fixtures["is_placeholder_match"] = ~(
    fixtures["team1_is_known"] & fixtures["team2_is_known"]
)

fixtures.groupby("stage")["is_placeholder_match"].agg(["sum", "count"])

,sum,count
stage,,
3rd Place Match,1,1
Final,1,1
Group Stage,0,72
Quarter-final,4,4
Round of 16,8,8
Round of 32,16,16
Semi-final,2,2


In [87]:
fixtures.loc[
    fixtures["is_placeholder_match"],
    ["stage", "team1", "team2", "is_placeholder_match"],
]

,stage,team1,team2,is_placeholder_match
72,Round of 32,1A,2B,True
73,Round of 32,1B,2A,True
74,Round of 32,1C,2D,True
75,Round of 32,1D,2C,True
76,Round of 32,1E,2F,True
77,Round of 32,1F,2E,True
78,Round of 32,1G,2H,True
79,Round of 32,1H,2G,True
80,Round of 32,1I,2J,True
81,Round of 32,1J,2I,True


In [88]:
known_fixture_teams = (
    set(fixtures.loc[fixtures["team1_is_known"], "team1"])
    | set(fixtures.loc[fixtures["team2_is_known"], "team2"])
)

old_team_names = {"Turkey", "United States", "Czech Republic"}
canonical_team_names = {"Türkiye", "USA", "Czechia"}

print("known fixture teams not in wc_2026_teams:")
print(sorted(known_fixture_teams - qualified_teams))
print("---" * 25)
print("wc_2026_teams not in known fixture teams:")
print(sorted(qualified_teams - known_fixture_teams))
print("---" * 25)
print("known fixture teams not in elo_latest:")
print(sorted(known_fixture_teams - elo_teams))
print("---" * 25)
print("elo_latest teams not in known fixture teams:")
print(sorted(elo_teams - known_fixture_teams))
print("---" * 25)
print("old team names present:")
print(sorted(old_team_names & (set(fixtures["team1"]) | set(fixtures["team2"]))))
print("---" * 25)
print("canonical team names present:")
print(sorted(canonical_team_names & (set(fixtures["team1"]) | set(fixtures["team2"]))))

known fixture teams not in wc_2026_teams:
[]
---------------------------------------------------------------------------
wc_2026_teams not in known fixture teams:
[]
---------------------------------------------------------------------------
known fixture teams not in elo_latest:
[]
---------------------------------------------------------------------------
elo_latest teams not in known fixture teams:
[]
---------------------------------------------------------------------------
old team names present:
[]
---------------------------------------------------------------------------
canonical team names present:
['Czechia', 'Türkiye', 'USA']


In [89]:
assert known_fixture_teams == qualified_teams
assert known_fixture_teams <= elo_teams
assert old_team_names.isdisjoint(set(fixtures["team1"]) | set(fixtures["team2"]))
assert canonical_team_names.issubset(set(fixtures["team1"]) | set(fixtures["team2"]))

print("Fixture team standardization checks passed.")

Fixture team standardization checks passed.


In [90]:
match_pair_key = fixtures.apply(
    lambda row: tuple(sorted([row["team1"], row["team2"]])),
    axis=1,
)

print("full duplicate rows:", fixtures.duplicated().sum())
print(
    "duplicate stage/date/team1/team2 rows:",
    fixtures.duplicated(["stage", "date", "team1", "team2"]).sum(),
)
print(
    "duplicate date/team1/team2 rows:",
    fixtures.duplicated(["date", "team1", "team2"]).sum(),
)
print(
    "duplicate unordered stage/date/team-pair rows:",
    fixtures.assign(match_pair_key=match_pair_key)
    .duplicated(["stage", "date", "match_pair_key"])
    .sum(),
)

assert fixtures.duplicated().sum() == 0
assert fixtures.duplicated(["stage", "date", "team1", "team2"]).sum() == 0
assert fixtures.duplicated(["date", "team1", "team2"]).sum() == 0
assert (
    fixtures.assign(match_pair_key=match_pair_key)
    .duplicated(["stage", "date", "match_pair_key"])
    .sum()
    == 0
)

print("Fixture duplicate checks passed.")

full duplicate rows: 0
duplicate stage/date/team1/team2 rows: 0
duplicate date/team1/team2 rows: 0
duplicate unordered stage/date/team-pair rows: 0
Fixture duplicate checks passed.


In [91]:
metadata_columns = [
    "team1_confederation",
    "team1_fifa_rank",
    "team1_coach",
    "team2_confederation",
    "team2_fifa_rank",
    "team2_coach",
]

known_matches = ~fixtures["is_placeholder_match"]
placeholder_matches = fixtures["is_placeholder_match"]

assert len(fixtures) == 104
assert known_matches.sum() == 72
assert placeholder_matches.sum() == 32
assert fixtures.loc[known_matches, metadata_columns].notna().all().all()
assert fixtures.loc[placeholder_matches, metadata_columns].isna().all().all()
assert fixtures.loc[known_matches, "group"].notna().all()
assert fixtures.loc[placeholder_matches, "group"].isna().all()

print("Fixture placeholder and expected-missing-value checks passed.")

Fixture placeholder and expected-missing-value checks passed.


In [92]:
venue_time_conflicts = fixtures[
    fixtures.duplicated(["date", "kickoff_et", "venue"], keep=False)
].sort_values(["date", "kickoff_et", "venue", "stage"])

print("venue/date/kickoff conflicts:", len(venue_time_conflicts))
venue_time_conflicts[["stage", "team1", "team2", "venue", "date", "kickoff_et"]]

venue/date/kickoff conflicts: 22


,stage,team1,team2,venue,date,kickoff_et
14,Group Stage,Brazil,Scotland,MetLife Stadium,2026-06-19,21:00 ET
15,Group Stage,Morocco,Haiti,MetLife Stadium,2026-06-19,21:00 ET
26,Group Stage,Germany,Ivory Coast,NRG Stadium,2026-06-20,19:00 ET
32,Group Stage,Netherlands,Sweden,NRG Stadium,2026-06-20,19:00 ET
4,Group Stage,Czechia,Mexico,Estadio Azteca,2026-06-24,21:00 ET
5,Group Stage,South Africa,South Korea,Estadio Azteca,2026-06-24,21:00 ET
16,Group Stage,Scotland,Brazil,Gillette Stadium,2026-06-24,21:00 ET
17,Group Stage,Morocco,Haiti,Gillette Stadium,2026-06-24,21:00 ET
28,Group Stage,Ecuador,Germany,AT&T Stadium,2026-06-25,18:00 ET
29,Group Stage,Curaçao,Ivory Coast,AT&T Stadium,2026-06-25,18:00 ET


In [93]:
assert fixtures["date"].notna().all()
assert fixtures["date"].min() == pd.Timestamp("2026-06-11")
assert fixtures["date"].max() == pd.Timestamp("2026-07-19")

print("Fixture validation checks passed.")

Fixture validation checks passed.
